In [ ]:
import os
from pathlib import Path

import h5py

import math
import torch
import random
import numpy as np

from operator import itemgetter 

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
dataset_name = "cora"
root_path = Path(f"outputs/{dataset_name}")

In [ ]:
model_name = ""
if dataset_name in ("tolokers2", "chameleon"):
    model_name = "megagat" 
elif dataset_name == "artnetviews":
    model_name = "megagcn"
elif dataset_name in ("cora", "citeseer"):
    model_name = "gcn"
elif dataset_name == "gapsmallqm9":
    model_name = "gcngraph"
    
dataset_version = "original"
if dataset_name == "tolokers2":
    experiment_name = "mordred_2026-03-27_19-04-29"
elif dataset_name == "artnetviews":
    experiment_name = "phoenix_2026-04-02_12-45-00"
elif dataset_name == "cora":
    experiment_name = "perceval_2026-03-02_15-34-05"
elif dataset_name == "citeseer":
    experiment_name = "arthur_2026-03-02_18-27-00"
elif dataset_name == "chameleon":
    dataset_version = "binary_features_logarithmic_target"
    experiment_name = "zeus_2026-04-02_15-42-51"
elif dataset_name == "gapsmallqm9":
    experiment_name = "gareth_2026-04-02_15-42-55"
    
complete_path = root_path / dataset_version / model_name / experiment_name / "h5files"

In [ ]:
import re
import yaml

In [ ]:
from inspector.models.mega_gcn import MegaGCNModel
from inspector.models.mega_gat import MegaGATModel
from inspector.models.gcn import GCNModel, GCNModelGraphWide

In [ ]:
from inspector.data.data_utils import StandardizeOutput, FeatureNormalisation, LogarithmOutput
from inspector.data.artnetviews_dataset import ArtnetViews
from inspector.data.tolokers2_dataset import Tolokers2
from inspector.data.chameleon_dataset import Chameleon
from inspector.data.smallqm9_dataset import GapSmallQM9

In [ ]:
from torch_geometric.datasets import Planetoid

from torch_geometric.transforms import Compose

In [ ]:
reduced_cfg_path = f"inspector/conf/model/{model_name}.yaml"

with open(reduced_cfg_path, "r") as f:
    info = yaml.safe_load(f)

if dataset_name in ("gapsmallqm9"):
    with open("inspector/conf/training/qm9_train.yaml", "r") as f:
        info_train = yaml.safe_load(f)

In [ ]:
if dataset_name == "artnetviews":
    pyg_data = ArtnetViews("./datasets/artnetviews/original", pre_transform=StandardizeOutput())
elif dataset_name == "tolokers2":
    pyg_data = Tolokers2("./datasets/tolokers2/original")
elif dataset_name == "cora":
    pyg_data = Planetoid("./datasets/cora/original", name="cora")
elif dataset_name == "citeseer":
    pyg_data = Planetoid("./datasets/citeseer/original", name="citeseer")
elif dataset_name == "chameleon":
    pyg_data = Chameleon("./datasets/chameleon/binary_features_logarithmic_target", pre_transform=Compose([LogarithmOutput(), StandardizeOutput()]))
elif dataset_name == "gapsmallqm9":
    pyg_data = GapSmallQM9("./datasets/gapsmallqm9/original", pre_transform=StandardizeOutput())

In [ ]:
exp = re.compile(r"(REP\d+-ENS\d)")

In [ ]:
done_exp_ids = set({})
test_metric = []

max_candidate_ingredients = 50
ingredients_state_dict = {}
ingredients_params = {}

for i, filename in enumerate(sorted(os.listdir(complete_path))):
    print(filename)
    if i+1 > max_candidate_ingredients:
        break
    if filename in done_exp_ids:
        continue
    try:
        exp_rep_id = exp.findall(filename)[0]
        metrics_file = "metrics_"+exp_rep_id+"_"+model_name+".hdf5"
        preds_file = "predictions_"+exp_rep_id+"_"+model_name+".hdf5"
        weights_file = "weights_"+exp_rep_id+"_"+model_name+".hdf5"
        
        done_exp_ids.add(exp_rep_id)
        fp_metrics = h5py.File(complete_path / metrics_file, "r")
        best_epoch_metrics = list(fp_metrics["metrics"].keys())[-1]
        test_metric.append(fp_metrics["metrics"][best_epoch_metrics]["validation_loss"][()])

        ########################
        
        fp_weights = h5py.File(complete_path / weights_file, "r")
        best_epoch_weights = list(fp_weights["weights"].keys())[-1]
        weights_group = fp_weights["weights"][best_epoch_weights]

        if model_name in ("megagcn", "megagat"):
            in_dim = weights_group['input_module.0.weight'].shape[1]
            hidden_dim = weights_group['input_module.0.weight'].shape[0]
            out_dim = weights_group['output_module.3.weight'].shape[0]
            num_layers = 3
        elif model_name == "gcn":
            in_dim = weights_group['layers.0.lin.weight'].shape[1]
            hidden_dim = weights_group['layers.0.lin.weight'].shape[0]
            out_dim = weights_group['layers.1.lin.weight'].shape[0]
        elif model_name == "gcngraph":
            in_dim = weights_group["gcn.layers.0.lin.weight"].shape[1]
            hidden_dim = weights_group["gcn.layers.0.lin.weight"].shape[0]
            out_dim = weights_group["regressor.lin2.weight"].shape[0]

        state_dict = {}
        for key in weights_group.keys():
            state_dict[key] = torch.tensor(weights_group[key][()])

        if model_name == "megagcn":
            model = MegaGCNModel(in_dim, 
                                 hidden_dim, 
                                 info["num_layers"], 
                                 out_dim, 
                                 act=info["act"], 
                                 jk=info["jk"], 
                                 dropout=info["dropout"]
                    )
            num_heads = -1
        elif model_name == "megagat":
            model = MegaGATModel(in_dim, 
                                 hidden_dim, 
                                 info["num_layers"], 
                                 out_dim, 
                                 act=info["act"], 
                                 jk=info["jk"], 
                                 dropout=info["dropout"],
                                 **{"heads": info["heads"]}
                    )
            num_heads = info["heads"]
        elif model_name == "gcn":
            model = GCNModel(
                in_dim,
                hidden_dim,
                info["num_layers"],
                out_dim,
                act=info["act"], 
                jk=info["jk"],
            )
        elif model_name == "gcngraph":
            model = GCNModelGraphWide(
                in_dim,
                hidden_dim,
                info["num_layers"],
                out_dim,
                act=info["act"], 
                jk=info["jk"],
                dropout=info["dropout"],
            )
            
        msg = model.load_state_dict(state_dict, strict=True)
        print(f"[PASSED] Successfully loaded weights: {msg}")
        print("Adding weights to the possible ingredients")
        
        ingredients_state_dict[i] = state_dict
        if model_name in ("megagcn", "megagat"):
            ingredients_params[i] = {
                "in_dim": in_dim,
                "hidden_dim": hidden_dim,
                "num_layers": info["num_layers"],
                "out_dim": out_dim,
                "act": info["act"],
                "jk": info["jk"],
                "dropout": info["dropout"],
                "heads": num_heads,
            }
        elif model_name == "gcn":
            ingredients_params[i] = {
                "in_dim": in_dim,
                "hidden_dim": hidden_dim,
                "num_layers": info["num_layers"],
                "out_dim": out_dim,
                "act": info["act"],
                "jk": info["jk"],
            }
        elif model_name == "gcngraph":
            ingredients_params[i] = {
                "in_dim": in_dim,
                "hidden_dim": hidden_dim,
                "num_layers": info["num_layers"],
                "out_dim": out_dim,
                "act": info["act"],
                "jk": info["jk"],
                "dropout": info["dropout"],
            }

    finally:
        fp_metrics.close()
        fp_weights.close()

In [ ]:
def _combination_at_rank(n: int, k: int, rank: int) -> tuple:
    """
    Return the combination at lexicographic position `rank` in C(n, k)
    using the combinatorial number system. O(k * n) worst case.

    Small example:
    If we ask for a combination with index 500000 of (50 5), for index 0 for the first index of the combination index
    we would get 211876 combinations. Not enough, we need more. If we try with index 1 for index 0 of the combination
    we are allowed to do an extra 194580.
    We need to get to the index 2 to be able to finally reach past 500000. So we pick 2 for index 0.
    If we were asking for combination 1000, since with index 0 for index 0 of the combination we can do
    211876 we definitely can use 0 and we will be able to reach the combination numbered 1000.
    This is only possible because we assume combinations are ordered in lexicographic order.

    """
    result = []
    start = 0
    for i in range(k, 0, -1):
        for c in range(start, n):
            ways = math.comb(n - c - 1, i - 1)
            if rank < ways:
                result.append(c)
                start = c + 1
                break
            rank -= ways
    return tuple(result)

In [ ]:
def _sample_ranks(total_combos: int, k: int, seed) -> list[int]:
    """Sample k distinct ranks from [0, total_combos) for arbitrarily large total_combos."""
    rng = random.Random(seed)
    seen = set()
    while len(seen) < k:
        r = rng.randint(0, total_combos - 1)
        seen.add(r)
    return list(seen)

In [ ]:
def make_the_soup(ingredient_weights, only_last_layer=False, index_for_rest=0):
    keys = list(ingredient_weights[0][1].keys())
    
    soup_weights = {}
    for key in keys:
        if (only_last_layer and key != keys[-1]):
            soup_weights[key] = ingredient_weights[index_for_rest][1][key]
            continue
            
        ll = [_weights[key] for _, _weights in ingredient_weights]
        stacked_ll = np.stack(ll)
        assert stacked_ll.shape[0] == len(ingredient_weights)

        soup_weights[key] = torch.from_numpy(stacked_ll.mean(axis=0))
        assert soup_weights[key].shape == ll[0].shape

    return soup_weights

def verify_params(ingredients_params):
    keys = set(ingredients_params[0][1].keys())

    soup_params = {}
    for key in keys:
        ll = [_params[key] for _, _params in ingredients_params]
        assert ll.count(ll[0]) == len(ll)
        soup_params[key] = ll[0]

    return soup_params

In [ ]:
def give_me_a_last_layer(ingredient_weights, my_index, rng):
    keys = list(ingredient_weights[0][1].keys())
    
    new_weights = {}
    chosen_index = my_index
    for key in keys:
        if key == keys[-1]:
            while chosen_index == my_index:
               chosen_index = rng.choice(len(ingredient_weights), 1)[0]
            new_weights[key] = ingredient_weights[chosen_index][1][key]
            continue
            
        new_weights[key] = ingredient_weights[my_index][1][key]

    return new_weights, chosen_index

In [ ]:
from torch_geometric.loader import DataLoader

In [ ]:
from inspector.metrics.metrics import average_precision, r2, compute_accuracy, classification_nll, diag_cov_nll, rmse

In [ ]:
def evaluate_model(weights, params, data, mask, metric_type="nll"):
    if model_name == "megagcn":
        model = MegaGCNModel(soup_params["in_dim"], 
                             soup_params["hidden_dim"], 
                             soup_params["num_layers"], 
                             soup_params["out_dim"], 
                             act=soup_params["act"], 
                             jk=soup_params["jk"], 
                             dropout=soup_params["dropout"]
                )
    elif model_name == "megagat":
        model = MegaGATModel(soup_params["in_dim"], 
                             soup_params["hidden_dim"], 
                             soup_params["num_layers"], 
                             soup_params["out_dim"], 
                             act=soup_params["act"], 
                             jk=soup_params["jk"], 
                             dropout=soup_params["dropout"],
                             **{"heads": soup_params["heads"]}
                )
    elif model_name == "gcn":
        model = GCNModel(soup_params["in_dim"], 
                     soup_params["hidden_dim"], 
                     soup_params["num_layers"], 
                     soup_params["out_dim"], 
                     act=soup_params["act"], 
                     jk=soup_params["jk"])

    msg = model.load_state_dict(weights, strict=True)
    print(f"[PASSED] Successfully loaded weights: {msg}")
    model.eval()
    with torch.no_grad():
        out, _ = model(data.x.double(), data.edge_index)
        
        # metrics_val = compute_accuracy(out[mask].argmax(dim=1), data.y[mask].squeeze())
        if dataset_name in ("cora", "citeseer", "tolokers2"):
            if dataset_name == "tolokers2":
                _out_prob = torch.sigmoid(out[mask].squeeze())
            else:
                _out_prob = torch.nn.functional.softmax(out[mask].squeeze(), dim=1)
                
            if metric_type == "nll":
                print(_out_prob.shape, data.y[mask].shape)
                metrics_val = classification_nll(_out_prob, data.y[mask])
            else:
                if dataset_name == "tolokers2":
                    # _out_class = torch.sigmoid(out).squeeze()[mask]  # <- ap
                    _out_class = (_out_prob > 0.5).type(torch.long).squeeze(-1)
                else:
                    _out_class = _out_prob.argmax(dim=1)
                metrics_val = compute_accuracy(_out_class, data.y[mask].squeeze())
                # metrics_val = average_precision(_out_class, data.y[mask])

        elif dataset_name in ("artnetviews", "chameleon"):
            mu = out[mask, 0].squeeze()
            s2 = out[mask, 1].squeeze()
            if metric_type == "nll":
                s2 = torch.nn.functional.softplus(s2) + 1e-6
                std_pred_total = s2.clamp(min=1e-10).sqrt().unsqueeze(-1)
                metrics_val = diag_cov_nll(mu, std_pred_total, data.y[mask])
            else:
                metrics_val = rmse(mu, data.y[mask].squeeze(), data.original_std)
                # metrics_val = r2(out[mask, 0].squeeze(), data.y[mask].squeeze())
    return metrics_val

In [ ]:
def evaluate_model_batched(weights, params, dataloader, original_std=None, metric_type="nll"):
    if model_name == "gcngraph":
        model = GCNModelGraphWide(
                in_dim,
                hidden_dim,
                info["num_layers"],
                out_dim,
                act=info["act"], 
                jk=info["jk"],
                dropout=info["dropout"],
        )

    msg = model.load_state_dict(weights, strict=True)
    print(f"[PASSED] Successfully loaded weights: {msg}")
    model.eval()
    
    running_metric = 0
    total_seen_examples = 0
    with torch.no_grad():
        for batch in dataloader:
            out, _ = model(batch.x.double(), 
                           batch.edge_index, 
                           edge_weight=batch.edge_attr.mean(dim=-1),
                           batch=batch.batch,
                     )
            n = len(batch)

            mu = out[:, 0].squeeze()
            s2 = out[:, 1].squeeze()
            if metric_type == "nll":
                s2 = torch.nn.functional.softplus(s2) + 1e-6
                std_pred_total = s2.clamp(min=1e-10).sqrt().unsqueeze(-1)
                running_metric += diag_cov_nll(mu, std_pred_total, batch.y) * n
            else:
                running_metric += rmse(mu, batch.y.squeeze(), original_std) * n
            total_seen_examples += n

    return running_metric / total_seen_examples

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
possible_ingredients = len(ingredients_state_dict)
actual_ingredients_to_use = 5
repetitions = 10

total_combos = math.comb(possible_ingredients, actual_ingredients_to_use)

sampled_ranks = _sample_ranks(
    total_combos, repetitions, 4242
)
combo_iter = [
    _combination_at_rank(possible_ingredients, actual_ingredients_to_use, r)
    for r in sampled_ranks
]

all_metrics = []
rng = np.random.default_rng(1111)
for _mt in ["nll", "point"]:
    metrics = []
    for i, combo in enumerate(combo_iter):
        indices = list(combo)
        print(indices)
        _ing = itemgetter(*indices)(list(ingredients_state_dict.items()))
        _params = itemgetter(*indices)(list(ingredients_params.items()))
    
        # Randomely switching last layers only
        # metrics = []
        # for j in range(len(indices)):
        #     soup_weights, chosen_index = give_me_a_last_layer(_ing, my_index=j, rng=rng)
        #     soup_params = verify_params(_params)
        
        #     metric = evaluate_model(soup_weights, soup_params, pyg_data, pyg_data.test_mask)
        #     metrics.append(metric)
        #     print(f"Metric: {metric}, Original NN index: {j}, Last layer NN index: {chosen_index}")
        # print(np.mean(metrics), np.std(metrics))
        
        # for j in range(len(indices)):  # <- soup only only_last_layer. It repeats the combo len(combo)*len(comb_iter) reps
        # index_for_rest=j
        soup_weights = make_the_soup(_ing, only_last_layer=False)
        soup_params = verify_params(_params)

        if dataset_name in ("gapsmallqm9"):
            test_loader = DataLoader(
                pyg_data[pyg_data.test_mask],
                batch_size=info_train["batch_size"],
                shuffle=False,
                num_workers=0,
            )
            metric = evaluate_model_batched(soup_weights, soup_params, test_loader, original_std=pyg_data.original_std, metric_type=_mt)
        else:
            metric = evaluate_model(soup_weights, soup_params, pyg_data, pyg_data.test_mask, metric_type=_mt)
        metrics.append(metric)
    
    print(np.mean(metrics), np.std(metrics))
    all_metrics.append(metrics)


In [ ]:
def get_stats_dict(data):
    """Calculates statistics and returns them in a list format for YAML."""
    return {
        "mean": [float(np.mean(data))],
        "std": [float(np.std(data))],
        "min": [float(np.min(data))],
        "max": [float(np.max(data))]
    }
    
def save_yaml(metric_results, metric_names, filename):
    results = {}
    for result, name in zip(metric_results, metric_names):
        results[name] = get_stats_dict(result)
    
    with open(f'{filename}.yaml', 'w') as file:
        yaml.dump(results, file, sort_keys=False, default_flow_style=False)

In [ ]:
dir_name = complete_path.parent / "soup"
os.makedirs(dir_name, exist_ok=True)
save_yaml(all_metrics, ["Diagonal NLL", "RMSE Test"], f"{dir_name}/nll_total_stats_soupx10")

---

In [ ]:
# individual_performances = []
# ingredient_items = list(ingredients_state_dict.items()) # List of (name, state_dict)

# print("Evaluating individual ingredients...")
# for i, (name, state_dict) in enumerate(ingredient_items):
#     # Use validation mask here!
#     print(f"[{i}/{len(ingredient_items)}]")
#     val_ap = evaluate_model(state_dict, ingredients_params[name], pyg_data, pyg_data.val_mask)
#     individual_performances.append((name, state_dict, val_ap))

# # Sort ingredients by validation performance (Descending)
# ranked_ingredients = sorted(individual_performances, key=lambda x: x[2], reverse=True)

# # Start the soup with the best individual model
# current_soup_ingredients = [(ranked_ingredients[0][0], ranked_ingredients[0][1])] # List of (name, state_dict)
# best_val_ap = ranked_ingredients[0][2]

# print(f"Starting soup with best model: {ranked_ingredients[0][0]} (Val AP: {best_val_ap:.4f})")

# for i in range(1, len(ranked_ingredients)):
#     candidate_name, candidate_state_dict, _ = ranked_ingredients[i]
    
#     # Create a potential soup
#     potential_ingredient_list = current_soup_ingredients + [(candidate_name, candidate_state_dict)]
#     test_weights = make_the_soup(tuple(potential_ingredient_list))
#     # (assuming they are all the same, uniform call above guarantees this)
#     master_params = ingredients_params[ranked_ingredients[0][0]]
    
#     current_val_ap = evaluate_model(test_weights, master_params, pyg_data, pyg_data.val_mask)
#     if current_val_ap > best_val_ap:
#         print(f"Found improvement: Adding {candidate_name}. New Val AP: {current_val_ap:.4f}")
#         best_val_ap = current_val_ap
#         current_soup_ingredients.append((candidate_name, candidate_state_dict))
#     else:
#         print(f"Skipping {candidate_name}: Did not improve performance.")

# final_soup_weights = make_the_soup(current_soup_ingredients)
# final_test_ap = evaluate_model(final_soup_weights, master_params, pyg_data, pyg_data.test_mask)

# print("-" * 30)
# print(f"Final Greedy Soup contains {len(current_soup_ingredients)} models.")
# print(f"Final Test AP: {final_test_ap:.4f}")